# 02 — Preprocessing & Feature Engineering

In [1]:
!pip install pandas



In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Build master DataFrame

In [3]:
!pip install requests


In [4]:
from src.data_loader import build_master
master = build_master(download=True)
print(master.shape)
master.head(3)

Loading JHU cases...
  Found cached: jhu_confirmed.csv


d:\Projects\EpiTrack\epidemic-spread-prediction\src\data_loader.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


Loading JHU deaths...
  Found cached: jhu_deaths.csv


d:\Projects\EpiTrack\epidemic-spread-prediction\src\data_loader.py:86: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


Loading OWID...
  Found cached: owid-covid-data.csv
  Master shape: (289762, 18)
(289762, 18)


,country,date,confirmed_cumulative,deaths_cumulative,new_tests_per_thousand,total_vaccinations_per_hundred,people_fully_vaccinated_per_hundred,stringency_index,population_density,median_age,hospital_beds_per_thousand,life_expectancy,population,mob_retail,mob_grocery,mob_transit,mob_work,mob_residential
0,Afghanistan,2020-01-22,0,0,NaN,NaN,NaN,0.0,54.42,18.6,0.5,64.83,41128772.0,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,2020-01-23,0,0,NaN,NaN,NaN,0.0,54.42,18.6,0.5,64.83,41128772.0,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,2020-01-24,0,0,NaN,NaN,NaN,0.0,54.42,18.6,0.5,64.83,41128772.0,NaN,NaN,NaN,NaN,NaN


## 2. Clean

In [5]:
from src.preprocessing import clean, save_processed, save_country_splits
cleaned = clean(master)
print(cleaned.dtypes)
print(cleaned.isnull().sum().sort_values(ascending=False).head(10))

d:\Projects\EpiTrack\epidemic-spread-prediction\src\preprocessing.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('country', group_keys=False).apply(_trim)


  After cleaning: (273203, 22)
country                                        object
date                                   datetime64[ns]
confirmed_cumulative                            int64
deaths_cumulative                               int64
new_tests_per_thousand                        float64
total_vaccinations_per_hundred                float64
people_fully_vaccinated_per_hundred           float64
stringency_index                              float64
population_density                            float64
median_age                                    float64
hospital_beds_per_thousand                    float64
life_expectancy                               float64
population                                    float64
mob_retail                                    float64
mob_grocery                                   float64
mob_transit                                   float64
mob_work                                      float64
mob_residential                               float

## 3. Feature engineering

In [6]:
from src.features import build_features, get_feature_columns
featured = build_features(cleaned)
print("Feature columns:", get_feature_columns())
featured[['country','date'] + get_feature_columns()[:6]].head()

  Building features...
  Feature matrix shape: (268982, 44)
Feature columns: ['new_cases_7d_lag7', 'new_cases_7d_lag14', 'new_cases_7d_lag21', 'new_cases_7d_rmean7', 'new_cases_7d_rstd7', 'new_cases_7d_rmean14', 'new_cases_7d_rstd14', 'new_cases_7d_rmean30', 'growth_rate', 'growth_accel', 'r0_estimate', 'cfr', 'day_of_week', 'month', 'days_since_start', 'population_density', 'median_age', 'hospital_beds_per_thousand', 'stringency_index', 'new_tests_per_thousand', 'vax_lag14']


,country,date,new_cases_7d_lag7,new_cases_7d_lag14,new_cases_7d_lag21,new_cases_7d_rmean7,new_cases_7d_rstd7,new_cases_7d_rmean14
0,Afghanistan,2020-03-06,0.0,0.0,0.0,0.000000,0.000000,0.000000
1,Afghanistan,2020-03-06,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,Afghanistan,2020-03-07,0.0,0.0,0.0,0.061224,0.161985,0.030612
3,Afghanistan,2020-03-07,0.0,0.0,0.0,0.122449,0.209121,0.061224
4,Afghanistan,2020-03-08,0.0,0.0,0.0,0.183673,0.229081,0.091837


## 4. Save outputs

In [7]:
# save_processed(featured, 'features.csv')
# save_country_splits(featured)
# print("Done.")
import os

def save_country_splits(df):
    split_dir = os.path.join('..', 'data', 'processed', 'country_splits')
    os.makedirs(split_dir, exist_ok=True)
    
    # Strip all Windows-illegal filename characters
    illegal = str.maketrans({
        ' ': '_', '/': '-', '*': '', ':': '-',
        '?': '', '"': '', '<': '', '>': '', '|': ''
    })
    
    for country, g in df.groupby('country'):
        safe = country.translate(illegal)
        g.to_csv(os.path.join(split_dir, f'{safe}.csv'), index=False)
    
    print(f"  Saved {df['country'].nunique()} country split files.")

save_country_splits(featured)
print("Done.")

  Saved 201 country split files.
Done.


## 5. Train / test split strategy
We use the last 30 days per country as the test set — a temporal holdout.

In [8]:
!pip install scikit-learn


In [9]:
from sklearn.model_selection import train_test_split

COUNTRIES = featured['country'].unique()
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'

def temporal_split(df, test_days=30):
    cutoff = df['date'].max() - pd.Timedelta(days=test_days)
    train = df[df['date'] <= cutoff]
    test  = df[df['date'] >  cutoff]
    return train, test

train, test = temporal_split(featured)
print(f"Train rows: {len(train):,}  |  Test rows: {len(test):,}")
print(f"Train ends: {train.date.max().date()}  Test starts: {test.date.min().date()}")

Train rows: 262,952  |  Test rows: 6,030
Train ends: 2023-02-07  Test starts: 2023-02-08
